# **TP3** 
# UC06 - Contrôles qualité

## Démarrage et chemins

In [1]:
import os
from pyspark.sql import SparkSession, functions as F
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TP3-Retail-Intelligence_Controles")
    .master("local[*]")
    .getOrCreate()
)

base = os.path.expanduser("~/Documents/IPSSI/DataLake Warehouse Lakehouse/TP/J3/data")

def lire(nom):
    return spark.read.parquet("file://" + base + "/" + nom)

jeux = ["silver/store_sales", "silver/item", "quarantine/store_sales", "quarantine/item",
        "gold/ca_par_categorie", "gold/ca_par_produit", "gold/ca_par_mois", "gold/ca_par_annee"]

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/19 14:04:33 WARN Utils: Your hostname, pauline-XMG-APEX-L23, resolves to a loopback address: 127.0.1.1; using 192.168.1.143 instead (on interface wlp4s0)
26/11/19 14:04:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 14:04:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/19 14:04:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/19 14:04:34 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


## Présence, format et taille

In [2]:
for nom in jeux:
    dossier = base + "/" + nom
    if not os.path.isdir(dossier):
        print(nom, "-> ABSENT")
        continue
    fichiers = os.listdir(dossier)
    parquet = [f for f in fichiers if f.endswith(".parquet")]
    taille = sum(os.path.getsize(os.path.join(dossier, f)) for f in fichiers)
    print(nom, "| fichiers .parquet :", len(parquet),
          "| _SUCCESS :", "_SUCCESS" in fichiers,
          "| taille :", round(taille / 1024 / 1024, 2), "Mo")

silver/store_sales | fichiers .parquet : 200 | _SUCCESS : True | taille : 64.79 Mo
silver/item | fichiers .parquet : 5 | _SUCCESS : True | taille : 3.52 Mo
quarantine/store_sales | fichiers .parquet : 1 | _SUCCESS : True | taille : 0.0 Mo
quarantine/item | fichiers .parquet : 1 | _SUCCESS : True | taille : 0.0 Mo
gold/ca_par_categorie | fichiers .parquet : 1 | _SUCCESS : True | taille : 0.0 Mo
gold/ca_par_produit | fichiers .parquet : 1 | _SUCCESS : True | taille : 0.2 Mo
gold/ca_par_mois | fichiers .parquet : 1 | _SUCCESS : True | taille : 0.0 Mo
gold/ca_par_annee | fichiers .parquet : 1 | _SUCCESS : True | taille : 0.0 Mo


## Lisibilité, lignes et schéma

In [3]:
for nom in jeux:
    df = lire(nom)
    print("=====", nom, "| lignes :", df.count())
    df.printSchema()

===== silver/store_sales | lignes : 667579
root
 |-- cle_date: integer (nullable = true)
 |-- _c1: integer (nullable = true)
 |-- id_produit: integer (nullable = true)
 |-- _c3: integer (nullable = true)
 |-- _c4: integer (nullable = true)
 |-- _c5: integer (nullable = true)
 |-- _c6: integer (nullable = true)
 |-- _c7: integer (nullable = true)
 |-- _c8: integer (nullable = true)
 |-- num_ticket: integer (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- _c11: double (nullable = true)
 |-- _c12: double (nullable = true)
 |-- prix_vente: double (nullable = true)
 |-- _c14: double (nullable = true)
 |-- _c15: double (nullable = true)
 |-- _c16: double (nullable = true)
 |-- _c17: double (nullable = true)
 |-- _c18: double (nullable = true)
 |-- _c19: double (nullable = true)
 |-- _c20: double (nullable = true)
 |-- _c21: double (nullable = true)
 |-- profit: double (nullable = true)
 |-- date_vente: date (nullable = true)
 |-- vente_a_perte: boolean (nullable = true)
 |-- p

## Contrôles qualité

In [4]:
sales = lire("silver/store_sales")
items = lire("silver/item")
nb = sales.count()

# Complétude : champs requis non nuls
for c in ["id_produit", "date_vente", "quantite", "prix_vente"]:
    print("Complétude", c, "-> nulls :", sales.filter(F.col(c).isNull()).count())
print("Complétude item.id_produit -> nulls :", items.filter(F.col("id_produit").isNull()).count())

# Unicité
print("Unicité ligne_id :", nb - sales.select("ligne_id").distinct().count(), "doublons")
print("Unicité item.id_produit :", items.count() - items.select("id_produit").distinct().count(), "doublons")

# Cohérence
print("Quantité <= 0 :", sales.filter(F.col("quantite") <= 0).count())
print("Prix de vente < 0 :", sales.filter(F.col("prix_vente") < 0).count())
print("Dates min / max :", sales.agg(F.min("date_vente"), F.max("date_vente")).first())

# Intégrité référentielle : ventes -> produits
orphelins = sales.join(items.select("id_produit"), on="id_produit", how="left_anti").count()
print("Ventes sans produit :", orphelins)

# Volumes : Silver + quarantaine = entrée
q = lire("quarantine/store_sales").count()
print("Silver + quarantaine :", nb + q, "(attendu : 667579)")

# Anomalies conservées (indicateurs)
print("Ventes à perte :", sales.filter("vente_a_perte").count())
print("Prix nuls :", sales.filter("prix_nul").count())

Complétude id_produit -> nulls : 0
Complétude date_vente -> nulls : 0
Complétude quantite -> nulls : 0
Complétude prix_vente -> nulls : 0
Complétude item.id_produit -> nulls : 0


Unicité ligne_id : 0 doublons
Unicité item.id_produit : 0 doublons
Quantité <= 0 : 0
Prix de vente < 0 : 0
Dates min / max : Row(min(date_vente)=datetime.date(2001, 1, 1), max(date_vente)=datetime.date(2005, 12, 13))
Ventes sans produit : 0
Silver + quarantaine : 667579 (attendu : 667579)
Ventes à perte : 495971
Prix nuls : 3324


## Cohérence Gold et Silver

In [5]:
total_silver = sales.agg(F.sum("_c15")).first()[0]

for nom in ["ca_par_categorie", "ca_par_produit", "ca_par_mois", "ca_par_annee"]:
    g = lire("gold/" + nom)
    somme = g.agg(F.sum("total_ventes")).first()[0]
    print(nom, "| écart avec Silver :", round(somme - total_silver, 2))

ca_par_categorie | écart avec Silver : 0.0
ca_par_produit | écart avec Silver : 0.0
ca_par_mois | écart avec Silver : 0.0
ca_par_annee | écart avec Silver : 0.0
